# 01 Smoke Test

Colab validation notebook for the first real D1 run.

- Repo remains the source of truth.
- Notebook bootstraps Colab, installs GroundingDINO and SAM2 in a Colab-friendly way, fetches smoke assets, and calls the repository CLI.
- Primary video path is the official SAM2 video predictor.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh


In [ ]:
%cd /content
!pip install -U ninja
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
!pip install transformers==4.46.3 accelerate==1.0.1 huggingface_hub==0.26.2 supervision==0.25.1


In [ ]:
%cd /content
!rm -rf GroundingDINO
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd /content/GroundingDINO
!git checkout v0.1.0-alpha2
!export CUDA_HOME=/usr/local/cuda && export FORCE_CUDA=1 && pip install --no-build-isolation -e .


In [ ]:
%cd /content
!rm -rf sam2
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!git checkout 2b90b9f5ceec907a1c18123530e92e794ad901a4
!pip uninstall -y SAM-2 sam2 || true
!export CUDA_HOME=/usr/local/cuda && pip install --no-build-isolation -e .


In [ ]:
import shutil
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
cuda_available = torch.cuda.is_available()
print('CUDA available:', cuda_available)

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None:
    print('GPU runtime not attached: nvidia-smi not found.')
else:
    gpu_name = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print('GPU:', gpu_name)

if not cuda_available:
    raise RuntimeError('Colab GPU runtime is not attached. Go to Runtime > Change runtime type > GPU, restart the session, then rerun from the top.')

import groundingdino
import sam2

print('GroundingDINO import OK:', groundingdino.__file__)
print('SAM2 import OK:', sam2.__file__)


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
INPUT_DIR = DRIVE_ROOT / 'inputs'
OUTPUT_ROOT = DRIVE_ROOT / 'results' / 'smoke_test'
GROUNDING_CKPT_URL = 'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth'
SAM2_CKPT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt'
GROUNDING_CKPT = CHECKPOINT_DIR / 'groundingdino_swint_ogc.pth'
SAM2_CKPT = CHECKPOINT_DIR / 'sam2.1_hiera_small.pt'

DEFAULT_IMAGE_URL = 'https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/.asset/cat_dog.jpeg'
DEFAULT_VIDEO_URL = 'https://raw.githubusercontent.com/IDEA-Research/Grounded-SAM-2/main/assets/hippopotamus.mp4'
DEFAULT_IMAGE_PATH = INPUT_DIR / 'groundingdino_cat_dog.jpeg'
DEFAULT_VIDEO_PATH = INPUT_DIR / 'grounded_sam2_hippopotamus.mp4'

IMAGE_PROMPT = 'dog'
VIDEO_PROMPT = 'hippopotamus'
MAX_FRAMES = 10
DEVICE = 'cuda'

for path in (CHECKPOINT_DIR, INPUT_DIR, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def download_if_missing(url: str, path: Path) -> None:
    if not path.exists():
        print(f'Downloading {path.name}...')
        urlretrieve(url, path)
    else:
        print(f'Already exists: {path}')

download_if_missing(GROUNDING_CKPT_URL, GROUNDING_CKPT)
download_if_missing(SAM2_CKPT_URL, SAM2_CKPT)
download_if_missing(DEFAULT_IMAGE_URL, DEFAULT_IMAGE_PATH)
download_if_missing(DEFAULT_VIDEO_URL, DEFAULT_VIDEO_PATH)

INPUT_IMAGE = str(DEFAULT_IMAGE_PATH)
INPUT_VIDEO = str(DEFAULT_VIDEO_PATH)
GROUNDING_CKPT = str(GROUNDING_CKPT)
SAM2_CKPT = str(SAM2_CKPT)

print('Image prompt:', IMAGE_PROMPT)
print('Video prompt:', VIDEO_PROMPT)
print('Image path:', INPUT_IMAGE)
print('Video path:', INPUT_VIDEO)
print('Grounding checkpoint:', GROUNDING_CKPT)
print('SAM2 checkpoint:', SAM2_CKPT)


In [ ]:
for required_path in (Path(GROUNDING_CKPT), Path(SAM2_CKPT)):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {required_path}')
    print(required_path, required_path.stat().st_size, 'bytes')


In [ ]:
%cd /content/ProjectFinalCS415
import shutil
import subprocess

IMAGE_OUTPUT = str(OUTPUT_ROOT / 'image')
shutil.rmtree(IMAGE_OUTPUT, ignore_errors=True)

cmd = [
    'python3', 'scripts/run_custom_video.py',
    '--config', 'configs/base.yaml',
    '--input_video', INPUT_IMAGE,
    '--prompt', IMAGE_PROMPT,
    '--output_dir', IMAGE_OUTPUT,
    '--grounding_ckpt', GROUNDING_CKPT,
    '--sam2_ckpt', SAM2_CKPT,
    '--device', DEVICE,
    '--max_frames', '1',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from IPython.display import Image, display

image_overlay = Path(IMAGE_OUTPUT) / 'smoke_image_overlay.png'
image_summary_path = Path(IMAGE_OUTPUT) / 'run_summary.json'

if not image_overlay.exists():
    raise FileNotFoundError(f'Missing image overlay: {image_overlay}')
if not image_summary_path.exists():
    raise FileNotFoundError(f'Missing image summary: {image_summary_path}')

display(Image(filename=str(image_overlay)))
with image_summary_path.open('r', encoding='utf-8') as handle:
    image_summary = json.load(handle)
for required_field in ('model_stack', 'runtime_sec', 'artifacts'):
    if required_field not in image_summary:
        raise KeyError(f'Missing required image summary field: {required_field}')
print(json.dumps(image_summary, indent=2))


In [ ]:
%cd /content/ProjectFinalCS415
import shutil
import subprocess

VIDEO_OUTPUT = str(OUTPUT_ROOT / 'video')
shutil.rmtree(VIDEO_OUTPUT, ignore_errors=True)

cmd = [
    'python3', 'scripts/run_custom_video.py',
    '--config', 'configs/base.yaml',
    '--input_video', INPUT_VIDEO,
    '--prompt', VIDEO_PROMPT,
    '--output_dir', VIDEO_OUTPUT,
    '--grounding_ckpt', GROUNDING_CKPT,
    '--sam2_ckpt', SAM2_CKPT,
    '--device', DEVICE,
    '--max_frames', str(MAX_FRAMES),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from IPython.display import Video, display

video_overlay = Path(VIDEO_OUTPUT) / 'smoke_video_overlay.mp4'
video_summary_path = Path(VIDEO_OUTPUT) / 'run_summary.json'

if not video_overlay.exists():
    raise FileNotFoundError(f'Missing video overlay: {video_overlay}')
if not video_summary_path.exists():
    raise FileNotFoundError(f'Missing video summary: {video_summary_path}')

display(Video(str(video_overlay), embed=True))
with video_summary_path.open('r', encoding='utf-8') as handle:
    video_summary = json.load(handle)
for required_field in ('model_stack', 'runtime_sec', 'artifacts'):
    if required_field not in video_summary:
        raise KeyError(f'Missing required video summary field: {required_field}')
print(json.dumps(video_summary, indent=2))
print('video_mode:', video_summary['artifacts'].get('video_mode'))
if 'fallback_reason' in video_summary['artifacts']:
    print('fallback_reason:', video_summary['artifacts']['fallback_reason'])
